<table align="center">
  <td align="center"><a target="_blank" href="http://introtodeeplearning.com">
        <img src="https://i.ibb.co/Jr88sn2/mit.png" style="padding-bottom:5px;" />
      Visit MIT Deep Learning</a></td>
  <td align="center"><a target="_blank" href="https://colab.research.google.com/github/MITDeepLearning/introtodeeplearning/blob/master/lab1/PT_Part1_Intro.ipynb">
        <img src="https://i.ibb.co/2P3SLwK/colab.png"  style="padding-bottom:5px;" />Run in Google Colab</a></td>
  <td align="center"><a target="_blank" href="https://github.com/MITDeepLearning/introtodeeplearning/blob/master/lab1/PT_Part1_Intro.ipynb">
        <img src="https://i.ibb.co/xfJbPmL/github.png"  height="70px" style="padding-bottom:5px;"  />View Source on GitHub</a></td>
</table>

# Copyright Information


In [ ]:
# Copyright 2026 MIT Introduction to Deep Learning. All Rights Reserved.
#
# Licensed under the MIT License. You may not use this file except in compliance
# with the License. Use and/or modification of this code outside of MIT Introduction
# to Deep Learning must reference:
#
# © MIT Introduction to Deep Learning
# http://introtodeeplearning.com
#

# PyTorch from First Principles

## Building Deep Learning Intuition One Tensor at a Time

This notebook builds the **complete mental model for PyTorch's tensor abstraction and autograd system** — starting with a running example that threads through every concept.

Every concept is demonstrated on the same problem:

> **Predicting house prices from size and age**  
> 5 houses, 2 features (sq ft, age) → 1 price. Small enough to visualise, real enough to matter.

| Part | Concept                    | Key Idea                                                                           |
| ---- | -------------------------- | ---------------------------------------------------------------------------------- |
| 1    | Tensors as Data Containers | Scalars → vectors → matrices → batches; faster + safer than lists                  |
| 2    | Computations on Tensors    | PyTorch traces operations into a graph; hand-tuning weights fails at scale         |
| 3    | Neural Networks in PyTorch | `nn.Module` wraps learnable `nn.Parameter`s; `forward()` defines the computation   |
| 4    | Automatic Differentiation  | `.backward()` computes ∂loss/∂every_weight in one call; gradient descent converges |
| 5    | From Toy to Production     | Same autograd scales from 3 params to 1.5 billion (GPT-2)                          |
| 6    | Music Generation Bonus     | Pre-trained transformers (MusicGen, TunesFormer) show inference at scale           |

---

## 0. Setup

[PyTorch](https://pytorch.org/) is a deep learning library known for flexibility and ease of use. For all labs in Introduction to Deep Learning 2026, a PyTorch version is available.


In [ ]:
import torch
import torch.nn as nn

# Download and import the MIT Introduction to Deep Learning package
!pip install mitdeeplearning --quiet
import mitdeeplearning as mdl

import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ── Deterministic seeds for reproducible results ──────────────────────────────
torch.manual_seed(42)
np.random.seed(42)
print("Seeds set → every run produces identical results.")

In [ ]:
# ── Our Running Example — House Price Prediction ──────────────────────────────
# Throughout this notebook, every concept is demonstrated on the same 5 houses.
houses = torch.tensor(
    [
        [1200.0, 10.0],  # size (sq ft), age (years)
        [1500.0, 5.0],
        [800.0, 15.0],
        [2000.0, 2.0],
        [1000.0, 12.0],
    ],
    dtype=torch.float32,
)
prices = torch.tensor([250.0, 320.0, 180.0, 450.0, 210.0])  # $1000s

print("Our running example — 5 houses:")
print("  Size (sq ft)  Age (yrs)  Price ($k)")
for i, (h, p) in enumerate(zip(houses, prices)):
    print(f"  [{i}]  {h[0]:6.0f}      {h[1]:4.0f}       {p:6.0f}")

# 2-panel scatter plot
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sc0 = axes[0].scatter(
    houses[:, 0], prices, c=houses[:, 1], cmap="viridis", s=100, edgecolor="black"
)
axes[0].set_xlabel("Size (sq ft)")
axes[0].set_ylabel("Price ($1000s)")
axes[0].set_title("Price vs Size (color = age)")
plt.colorbar(sc0, ax=axes[0], label="Age (years)")
sc1 = axes[1].scatter(
    houses[:, 1], prices, c=houses[:, 0], cmap="plasma", s=100, edgecolor="black"
)
axes[1].set_xlabel("Age (years)")
axes[1].set_ylabel("Price ($1000s)")
axes[1].set_title("Price vs Age (color = size)")
plt.colorbar(sc1, ax=axes[1], label="Size (sq ft)")
plt.suptitle("Our Running Example: 5 Houses → 1 Price", fontweight="bold")
plt.tight_layout()
plt.show()
print(
    "This dataset is our 'the cat sat on the mat' — every concept demonstrated on these 5 houses."
)

---

## Part 1 — Tensors as Data Containers

PyTorch is a machine learning library, like TensorFlow. At its core, PyTorch provides an interface for creating and manipulating [tensors](https://pytorch.org/docs/stable/tensors.html), which are data structures that you can think of as multi-dimensional arrays. Tensors are represented as n-dimensional arrays of base datatypes such as a string or integer -- they provide a way to generalize vectors and matrices to higher dimensions. PyTorch provides the ability to perform computation on these tensors, define neural networks, and train them efficiently.

The [`shape`](https://pytorch.org/docs/stable/generated/torch.Tensor.shape.html#torch.Tensor.shape) of a PyTorch tensor defines its number of dimensions and the size of each dimension. The `ndim` or [`dim`](https://pytorch.org/docs/stable/generated/torch.Tensor.dim.html#torch.Tensor.dim) of a PyTorch tensor provides the number of dimensions (n-dimensions) -- this is equivalent to the tensor's rank (as is used in TensorFlow), and you can also think of this as the tensor's order or degree.

Let's start by creating some tensors and inspecting their properties:


#### **Predict first** — why do we need tensors?

Python has lists. NumPy has arrays. Predict which of these are true about `torch.Tensor` vs nested lists:

1. **Speed**: Can tensors outrun nested lists on matrix multiply?
2. **Shape safety**: Will tensors catch a shape mismatch that lists silently swallow?
3. **GPU support**: Can a list of lists run on a GPU?

Make your prediction, then run the next cell.


In [ ]:
# ── WHY tensors? Three reasons, measured ──────────────────────────────────────
import time

# Reason 1: Speed — matrix multiply
n = 1000
np_a = np.random.randn(n, n).astype(np.float32)
np_b = np.random.randn(n, n).astype(np.float32)
t0 = time.time()
_ = np_a @ np_b
np_time = time.time() - t0

t_a = torch.from_numpy(np_a)
t_b = torch.from_numpy(np_b)
t0 = time.time()
_ = torch.matmul(t_a, t_b)
torch_time = time.time() - t0

print(f"Matrix multiply (1000×1000):")
print(f"  NumPy:  {np_time*1000:.1f} ms")
print(
    f"  PyTorch: {torch_time*1000:.1f} ms  ({np_time/max(torch_time,1e-9):.1f}× speedup)"
)

# Reason 2: Shape safety
try:
    bad = torch.randn(3, 4) + torch.randn(5, 6)
except RuntimeError as e:
    print(f"\n✓ Shape mismatch caught: '{e}'")
    print("  Lists would silently fail or give wrong results.")

# Reason 3: GPU support
print(f"\nGPU available: {torch.cuda.is_available()}")
print("  → torch.Tensor can move to GPU with .to('cuda'); lists cannot.")
print("\n→ Tensors are PURPOSE-BUILT for deep learning: fast, safe, GPU-ready.")

In [ ]:
# ── Scalar tensors (0-D) ──────────────────────────────────────────────────────
integer = torch.tensor(1234)
decimal = torch.tensor(3.14159265359)

print(f"`integer` is a {integer.ndim}-d Tensor: {integer}")
print(f"`decimal` is a {decimal.ndim}-d Tensor: {decimal}")

Vectors and lists can be used to create 1-d tensors:


In [ ]:
# ── 1-D Tensors (vectors) ─────────────────────────────────────────────────────
fibonacci = torch.tensor([1, 1, 2, 3, 5, 8])
count_to_100 = torch.tensor(range(100))

print(f"`fibonacci` is a {fibonacci.ndim}-d Tensor with shape: {fibonacci.shape}")
print(
    f"`count_to_100` is a {count_to_100.ndim}-d Tensor with shape: {count_to_100.shape}"
)

Next, let’s create 2-d (i.e., matrices) and higher-rank tensors. In image processing and computer vision, we will use 4-d Tensors with dimensions corresponding to batch size, number of color channels, image height, and image width.


In [ ]:
# ── 2-D and higher-rank Tensors ───────────────────────────────────────────────

'''TODO: Define a 2-d Tensor'''
matrix = # TODO

assert isinstance(matrix, torch.Tensor), "matrix must be a torch Tensor object"
assert matrix.ndim == 2

'''TODO: Define a 4-d Tensor.'''
# Use torch.zeros to initialize a 4-d Tensor of zeros with size 10 x 3 x 256 x 256.
#   You can think of this as 10 images where each image is RGB 256 x 256.
images = # TODO

assert isinstance(images, torch.Tensor), "images must be a torch Tensor object"
assert images.ndim == 4, "images must have 4 dimensions"
assert images.shape == (10, 3, 256, 256), "images is incorrect shape"
print(f"images is a {images.ndim}-d Tensor with shape: {images.shape}")

As you have seen, the `shape` of a tensor provides the number of elements in each tensor dimension. The `shape` is quite useful, and we'll use it often. You can also use slicing to access subtensors within a higher-rank tensor:


In [ ]:
# ── Tensor slicing — accessing sub-tensors ────────────────────────────────────
row_vector = matrix[1]
column_vector = matrix[:, 1]
scalar = matrix[0, 1]

print(f"`row_vector`: {row_vector}")
print(f"`column_vector`: {column_vector}")
print(f"`scalar`: {scalar}")

#### What just happened — and what's missing

Tensors are PyTorch's data container: fast (GPU-ready), safe (shape-checked), and built for batches. Every model input (images, text, audio) becomes a tensor before processing.

**Missing piece**: We have the _container_, but we haven't built anything that _learns_ yet. For that, we need operations that PyTorch can differentiate — next.


---

## Part 2 — Computations on Tensors

A convenient way to think about and visualize computations in a machine learning framework like PyTorch is in terms of graphs. We can define this graph in terms of tensors, which hold data, and the mathematical operations that act on these tensors in some order. Let's look at a simple example, and define this computation using PyTorch:

![alt text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/add-graph.png)


In [ ]:
# ── Simple computation graph — node addition ──────────────────────────────────
# Create the nodes in the graph and initialize values
a = torch.tensor(15)
b = torch.tensor(61)

# Add them!
c1 = torch.add(a, b)
c2 = a + b  # PyTorch overrides the "+" operation so that it is able to act on Tensors
print(f"c1: {c1}")
print(f"c2: {c2}")

Notice how we've created a computation graph consisting of PyTorch operations, and how the output is a tensor with value 76 -- we've just created a computation graph consisting of operations, and it's executed them and given us back the result.

Now let's consider a slightly more complicated example:

![alt text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/computation-graph.png)

Here, we take two inputs, `a, b`, and compute an output `e`. Each node in the graph represents an operation that takes some input, does some computation, and passes its output to another node.

Let's define a simple function in PyTorch to construct this computation function:


In [ ]:
# ── Multi-step computation graph: a,b → c,d → e ──────────────────────────────

# Construct a simple computation function
def func(a, b):
    '''TODO: Define the operation for c, d, e.'''
    c = # TODO
    d = # TODO
    e = # TODO
    return e

Now, we can call this function to execute the computation graph given some inputs `a,b`:


In [ ]:
# ── Execute the computation graph ─────────────────────────────────────────────
# Consider example values for a,b
a, b = 1.5, 2.5
# Execute the computation
e_out = func(a, b)
print(f"e_out: {e_out}")

#### What just happened — and what's missing

PyTorch executed our computation and returned a tensor. But it did more than arithmetic — it **recorded every operation** in a computation graph. When we later call `.backward()` on any output, PyTorch walks that graph in reverse to compute gradients for every input that has `requires_grad=True`.

**Missing piece**: We defined _what to compute_, but not _what to optimise_. For that we need a trainable model with learnable weights — that's `nn.Module`, next.


In [ ]:
# ── Computation on our house dataset — price prediction by hand ───────────────
size = houses[0, 0]  # 1200 sq ft
age = houses[0, 1]  # 10 years
true_price = prices[0]  # $250k

# Hand-picked weights (we'll learn better ones later via gradient descent)
w_size = torch.tensor(0.15)
w_age = torch.tensor(-5.0)
bias = torch.tensor(100.0)

contrib_size = w_size * size
contrib_age = w_age * age
price_pred = contrib_size + contrib_age + bias

print(f"House [0]: {size:.0f} sq ft, {age:.0f} years → true price ${true_price:.0f}k")
print(f"  w_size × size = {w_size:.2f} × {size:.0f} = ${contrib_size:.1f}k")
print(f"  w_age  × age  = {w_age:.2f} × {age:.0f} = ${contrib_age:.1f}k")
print(f"  + bias        = ${bias:.1f}k")
print(
    f"  → Predicted:  ${price_pred:.1f}k   (error: ${abs(price_pred - true_price):.1f}k)"
)
print(
    "\nPyTorch traced this computation automatically. We'll use that trace for autograd next."
)

### Exercise — manual weight tuning

Before Section 1.3 introduces _learnable_ weights, try tuning the three weights by hand.

**Predict**: Can you fit all 5 houses within ±$10k error just by adjusting numbers?


In [ ]:
# 🧪 EXERCISE — manual weight tuning
# 👉 CHANGE these three weights to predict all 5 house prices within ±$10k error
w_size = torch.tensor(0.15)  # ← try 0.10, 0.20, 0.25...
w_age = torch.tensor(-5.0)  # ← try -3.0, -8.0...
bias = torch.tensor(100.0)  # ← try 50.0, 150.0...

predictions = houses[:, 0] * w_size + houses[:, 1] * w_age + bias
errors = torch.abs(predictions - prices)

print("Manual tuning results:")
print(f"  {'House':>6}  {'True':>8}  {'Predicted':>10}  {'Error':>8}")
for i, (p_true, p_pred, err) in enumerate(zip(prices, predictions, errors)):
    check = "✓" if err < 10.0 else "✗"
    print(f"  [{i}]     ${p_true:6.1f}k   ${p_pred:6.1f}k      ${err:5.1f}k  {check}")
mean_error = errors.mean()
print(f"\nMean absolute error: ${mean_error:.1f}k")
if mean_error < 10.0:
    print("✓ You nailed it by hand — but imagine 100 features, 10000 houses...")
else:
    print(
        "→ Hand-tuning fails even on 5 houses. We need LEARNING: autograd + gradient descent."
    )

#### What just happened — and the crack it leaves open

We manually wrote `w_size * size + w_age * age + bias` and saw the prediction fail. But:

- We **hand-picked** those weights. What if we have 100 features?
- We **guessed** they'd work. How do we find _good_ weights systematically?

That's the job of **gradient descent** — and it requires computing `∂loss/∂w_size`. That's what autograd does automatically.


---

## Part 3 — Neural Networks in PyTorch

A perceptron does one thing: take a set of inputs, multiply each by a learned weight, sum them up, then squash the result through a non-linearity so the output stays bounded. The formula shorthand for this is $y = \sigma(Wx + b)$.

The intuition behind each piece: **$W$ decides how much each feature matters** — a large positive weight for `size` means bigger houses cost more; **$b$ is a constant offset** (a baseline price even for the smallest house); **$\sigma$ (sigmoid) keeps output in $(0, 1)$** — without it, stacking multiple layers would just collapse into one giant linear function, which defeats the point. Every arrow in the diagram below is one entry in $W$ or $b$, and those numbers are exactly what the model _learns_.

![alt text](https://raw.githubusercontent.com/MITDeepLearning/introtodeeplearning/master/lab1/img/computation-graph-2.png)

PyTorch's [`torch.nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) is the container that holds these learnable parameters and wires up the computation. You subclass it, declare weights as `nn.Parameter` objects, and override `forward()` with the actual math. PyTorch then traces every operation through those parameters so `.backward()` can compute gradients for each one automatically — no manual calculus needed.

Let's write a dense layer class to implement the perceptron above.


#### **Predict first** — what does `nn.Module` give us?

We're about to define a custom dense layer by subclassing `torch.nn.Module`. Before reading the class, predict which of these is true:

1. We must manually call `.backward()` on **each** `nn.Parameter` separately inside the class.
2. Wrapping a tensor in `nn.Parameter` is sufficient — autograd tracks it automatically from that point forward, no extra code needed.
3. We need a custom gradient-accumulation loop inside `__init__` for parameters to work.

Pick your answer, then read and run the class definition below.


In [ ]:
# ── Hand-written dense layer (nn.Module subclass) ─────────────────────────────
# num_inputs: number of input nodes
# num_outputs: number of output nodes
# x: input to the layer

class OurDenseLayer(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(OurDenseLayer, self).__init__()
        # Define and initialize parameters: a weight matrix W and bias b
        # Note that the parameter initialize is random!
        self.W = torch.nn.Parameter(torch.randn(num_inputs, num_outputs))
        self.bias = torch.nn.Parameter(torch.randn(num_outputs))

    def forward(self, x):
        '''TODO: define the operation for z (hint: use torch.matmul).'''
        z = # TODO

        '''TODO: define the operation for out (hint: use torch.sigmoid).'''
        y = # TODO
        return y

Now, let's test the output of our layer.


In [ ]:
# ── Test OurDenseLayer ────────────────────────────────────────────────────────
# Define a layer and test the output!
num_inputs = 2
num_outputs = 3
layer = OurDenseLayer(num_inputs, num_outputs)
x_input = torch.tensor([[1, 2.0]])
y = layer(x_input)

print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

Conveniently, PyTorch has defined a number of `nn.Modules` (or Layers) that are commonly used in neural networks, for example a [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) or [`nn.Sigmoid`](https://pytorch.org/docs/stable/generated/torch.nn.Sigmoid.html) module.

Now, instead of using a single `Module` to define our simple neural network, we'll use the [`nn.Sequential`](https://pytorch.org/docs/stable/generated/torch.nn.Sequential.html) module from PyTorch and a single [`nn.Linear` ](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) layer to define our network. With the `Sequential` API, you can readily create neural networks by stacking together layers like building blocks.


In [ ]:
# ── Same layer via nn.Sequential (no subclassing needed) ──────────────────────

# define the number of inputs and outputs
n_input_nodes = 2
n_output_nodes = 3

# Define the model
"""TODO: Use the Sequential API to define a neural network with a
    single linear (dense!) layer, followed by non-linearity to compute z"""
model = nn.Sequential(""" TODO """)

We've defined our model using the Sequential API. Now, we can test it out using an example input:


In [ ]:
# ── Test Sequential model ─────────────────────────────────────────────────────
# Test the model with example input
x_input = torch.tensor([[1, 2.0]])
model_output = model(x_input)
print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

With PyTorch, we can create more flexible models by subclassing [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html). The `nn.Module` class allows us to group layers together flexibly to define new architectures.

As we saw earlier with `OurDenseLayer`, we can subclass `nn.Module` to create a class for our model, and then define the forward pass through the network using the `forward` function. Subclassing affords the flexibility to define custom layers, custom training loops, custom activation functions, and custom models. Let's define the same neural network model as above (i.e., Linear layer with an activation function after it), now using subclassing and using PyTorch's built in linear layer from `nn.Linear`.


In [ ]:
# ── Custom forward pass via subclassing nn.Module ─────────────────────────────


class LinearWithSigmoidActivation(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(LinearWithSigmoidActivation, self).__init__()
        """TODO: define a model with a single Linear layer and sigmoid activation."""
        self.linear = """TODO: linear layer"""
        self.activation = """TODO: sigmoid activation"""

    def forward(self, inputs):
        linear_output = self.linear(inputs)
        output = self.activation(linear_output)
        return output

Let's test out our new model, using an example input, setting `n_input_nodes=2` and `n_output_nodes=3` as before.


In [ ]:
# ── Test LinearWithSigmoidActivation ──────────────────────────────────────────
n_input_nodes = 2
n_output_nodes = 3
model = LinearWithSigmoidActivation(n_input_nodes, n_output_nodes)
x_input = torch.tensor([[1, 2.0]])
y = model(x_input)
print(f"input shape: {x_input.shape}")
print(f"output shape: {y.shape}")
print(f"output result: {y}")

#### What just happened — and what's missing

`nn.Module` bundles **weights + computation**: declare once, call repeatedly. PyTorch registers every `nn.Parameter` automatically — no manual bookkeeping.

**Missing piece**: The weights were _randomly initialised_ — the model outputs nonsense. We need a way to measure how wrong it is (**loss function**) and a way to systematically improve the weights (**gradient descent**). That's `autograd` — next in Part 4.


Importantly, `nn.Module` affords us a lot of flexibility to define custom models. For example, we can use boolean arguments in the `forward` function to specify different network behaviors, for example different behaviors during training and inference. Let's suppose under some instances we want our network to simply output the input, without any perturbation. We define a boolean argument `isidentity` to control this behavior:


In [ ]:
# ── Custom behavior: conditional identity pass ────────────────────────────────


class LinearButSometimesIdentity(nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super(LinearButSometimesIdentity, self).__init__()
        self.linear = nn.Linear(num_inputs, num_outputs)

    """TODO: Implement the behavior where the network outputs the input, unchanged,
        under control of the isidentity argument."""

    def forward(self, inputs, isidentity=False):
        """TODO"""

Let's test this behavior:


In [ ]:
# ── Test LinearButSometimesIdentity ───────────────────────────────────────────
# Test the IdentityModel
model = LinearButSometimesIdentity(num_inputs=2, num_outputs=3)
x_input = torch.tensor([[1, 2.]])

'''TODO: pass the input into the model and call with and without the input identity option.'''
out_with_linear = # TODO

out_with_identity = # TODO

print(f"input: {x_input}")
print("Network linear output: {}; network identity output: {}".format(out_with_linear, out_with_identity))

Now that we have learned how to define layers and models in PyTorch using both the Sequential API and subclassing `nn.Module`, we're ready to turn our attention to how to actually implement network training with backpropagation.


In [ ]:
# ── HousePriceModel: the same prediction, but with LEARNABLE weights ──────────
class HousePriceModel(torch.nn.Module):
    """Predicts house price from size and age using a single linear layer."""

    def __init__(self):
        super().__init__()
        self.w_size = torch.nn.Parameter(torch.randn(1) * 0.01)
        self.w_age = torch.nn.Parameter(torch.randn(1) * 0.01)
        self.bias = torch.nn.Parameter(torch.randn(1) * 0.01)

    def forward(self, size, age):
        return self.w_size * size + self.w_age * age + self.bias


torch.manual_seed(42)
model = HousePriceModel()
print("Model parameters (random initialization):")
for name, param in model.named_parameters():
    print(f"  {name}: {param.item():.4f}")

pred = model(houses[0, 0], houses[0, 1])
print(
    f"\nPrediction for house [0]: ${pred.item():.1f}k  (random weights → bad prediction)"
)
print("Next: we'll use autograd to LEARN better weights automatically.")

---

## Part 4 — Automatic Differentiation

Here is the core problem that makes neural networks trainable: we have a single loss number, and we need to know which direction to nudge _each_ of potentially millions of weights to make that number smaller. Computing those partial derivatives by hand — even for our 3-weight house-price model — is tedious algebra. For millions of weights it is impossible.

PyTorch solves this with [`torch.autograd`](https://pytorch.org/docs/stable/autograd.html): every time you do math on a tensor that has `requires_grad=True`, PyTorch silently records the operation in a computation graph. When you later call `.backward()` on the final loss, PyTorch walks that graph in reverse (applying the chain rule layer by layer) and deposits `∂loss/∂param` into each parameter's `.grad` attribute — all in one pass, regardless of how many parameters exist. This is [backpropagation](https://en.wikipedia.org/wiki/Backpropagation), automated.

To see the mechanism at its clearest, let's verify it on $y = x^2$ — one input, one output, derivative known analytically — before applying it to a full model:


#### **Predict first** — what does `.backward()` return?

We're about to compute the derivative of $y = x^2$ at $x = 3.0$ using PyTorch's autograd. Predict the result before running:

1. `dy_dx = 9.0` — because $y = x^2 = 9$ at $x = 3$.
2. `dy_dx = 6.0` — because $\frac{dy}{dx} = 2x$, evaluated at $x = 3$.
3. `dy_dx = 3.0` — because $x = 3$.

Only one of these is the gradient. Pick your answer, then run the cell.


In [ ]:
# ── Gradient of y = x² at x = 3 ──────────────────────────────────────────────
# requires_grad=True tells PyTorch to record all operations on x in a graph
x = torch.tensor(3.0, requires_grad=True)
y = x**2
y.backward()  # Walk the graph in reverse: dy/dx = 2x

dy_dx = x.grad
print("dy_dx of y=x^2 at x=3.0 is: ", dy_dx)
# dy/dx = 2x → at x=3: dy_dx = 6
assert dy_dx == 6.0
print("  → .backward() computed the analytic derivative automatically.")

The derivative alone does not train anything — you also need something to minimise. In neural networks that is the **loss function**: a single number measuring how wrong the current weights are. The gradient tells you the local slope of that loss surface; gradient descent says "take a small step downhill."

To build the intuition without any distractions, we will minimise $L = (x - x_f)^2$ — a parabola with a known bottom at $x_f$. We could solve it analytically (the minimum is trivially $x = x_f$), but we deliberately will _not_: instead, autograd computes $\partial L / \partial x$ at each iteration and gradient descent steps toward the bottom. This is the **exact same loop** that trains every neural network — just on one scalar instead of millions of weights.


In [ ]:
# ── Gradient descent: minimize L = (x − x_f)² ────────────────────────────────
# Minimizing L = (x - x_f)^2 analytically gives x = x_f.
# Here we let gradient descent find that minimum iteratively — same mechanism
# that trains neural networks, just on a single scalar instead of millions of weights.

# Initialize a random value for our initial x
x = torch.randn(1)
print(f"Initializing x={x.item()}")

learning_rate = 1e-2  # Learning rate
history = []
x_f = 4  # Target value


# We will run gradient descent for a number of iterations. At each iteration, we compute the loss,
#   compute the derivative of the loss with respect to x, and perform the update.
for i in range(500):
    x = torch.tensor([x], requires_grad=True)

    # TODO: Compute the loss as the square of the difference between x and x_f
    loss = # TODO

    # Backpropagate through the loss to compute gradients
    loss.backward()

    # Update x with gradient descent
    x = x.item() - learning_rate * x.grad

    history.append(x.item())

# Plot the evolution of x as we optimize toward x_f!
plt.plot(history)
plt.plot([0, 500], [x_f, x_f])
plt.legend(('Predicted', 'True'))
plt.xlabel('Iteration')
plt.ylabel('x value')
plt.show()
print(f"  → x converged to {history[-1]:.4f}; target was {x_f}.")
print("  → Same loop — gradient × learning rate — trains every neural network.")

In [ ]:
# ── Visualizing what .backward() actually does ────────────────────────────────
torch.manual_seed(42)
model_viz = HousePriceModel()
optimizer = torch.optim.SGD(model_viz.parameters(), lr=1e-5)

optimizer.zero_grad()
preds = model_viz(houses[:, 0], houses[:, 1])
loss = torch.mean((preds - prices) ** 2)
loss.backward()

print(f"After loss.backward(), gradients computed:")
for name, param in model_viz.named_parameters():
    print(f"  ∂loss/∂{name}: {param.grad.item():+.4f}")

print(f"\nLoss: {loss.item():.2f}")
print("→ .backward() computed ∂loss/∂w_size, ∂loss/∂w_age, ∂loss/∂bias automatically.")
print("  Gradient descent updates each weight in the direction that reduces loss.")

# Show one update step
print(f"\nBefore update: w_size = {model_viz.w_size.item():.4f}")
optimizer.step()
print(
    f"After step:    w_size = {model_viz.w_size.item():.4f}  (moved toward lower loss)"
)

#### What just happened — from toy to production

We trained a 3-parameter model with `.backward()` and watched loss drop. The same mechanism trains:

- GPT-2: 1.5 billion parameters
- Stable Diffusion: 860 million parameters

**The only difference is scale.** The autograd graph, the gradient computation, the optimizer step — all identical.


---

## Part 5 — From Toy to Production: Same Machinery, Bigger Numbers

Our house-price model has **3 learnable parameters**. A production deep-learning model has millions — but the _mechanism_ is identical: `nn.Parameter`, `.forward()`, `.backward()`, `optimizer.step()`.

| Model                            | Parameters   | Same autograd? | Same nn.Module? |
| -------------------------------- | ------------ | -------------- | --------------- |
| Our toy (house prices)           | 3            | ✓              | ✓               |
| ResNet-18 (image classification) | 11.7 million | ✓              | ✓               |
| GPT-2 (language model)           | 1.5 billion  | ✓              | ✓               |
| Stable Diffusion                 | 860 million  | ✓              | ✓               |

**The only difference is scale.** If you understood gradient descent on 3 weights, you understand it on 860 million.


In [ ]:
# ── Toy-to-real: same nn.Module, different scale ──────────────────────────────
try:
    import torchvision

    real_model = torchvision.models.resnet18(weights=None)  # structure only
    total = sum(p.numel() for p in real_model.parameters())
    trainable = sum(p.numel() for p in real_model.parameters() if p.requires_grad)
    print(f"ResNet-18 architecture:")
    print(f"  Total parameters   : {total:,}")
    print(f"  Trainable parameters: {trainable:,}")
    print(f"  Layers: {len(list(real_model.children()))}")
    print(
        "\nEvery one of those 11.7M parameters is a torch.nn.Parameter, just like our w_size."
    )
    print(
        "Every forward pass traces a computation graph. Every .backward() computes all gradients."
    )
    print(
        "The machinery you learned on 3 weights scales to billions — no new concepts needed."
    )
except ImportError:
    print(
        "[torchvision not installed — the point stands: production models use the same"
    )
    print(
        " nn.Module / autograd machinery you just learned on 5 houses and 3 weights.]"
    )

Parts 1–5 cover the complete PyTorch mental model — tensors, graphs, modules, autograd, and gradient descent. Part 6 applies the same autograd machinery at production scale via pre-trained music generation models.


---

## Part 6 — Music Generation with HuggingFace

Instead of training an RNN from scratch, this section lets you pick a
**pre-trained model from HuggingFace Hub** and generate music immediately.

Two models are available, selectable in the next cell:

| Model                     | Approach            | Input                 | Output            |
| ------------------------- | ------------------- | --------------------- | ----------------- |
| `facebook/musicgen-small` | Transformer seq2seq | **text prompt**       | raw audio (WAV)   |
| `sander-wood/tunesformer` | Causal LM           | **ABC notation seed** | ABC notation text |

> **ABC notation** is a text-based music format used for Irish/Celtic folk tunes.
> Example: `X:1\nT:Title\nM:6/8\nK:Gmaj\n|: G2A B2c | d2e fed |`
> The model extends that seed, producing a complete tune you can play with `music21` or `abc2midi`.


### Why Pre-trained Models Instead of Training From Scratch?

You've learned PyTorch's autograd system on a 3-parameter toy model. Real-world models (GPT, BERT, MusicGen) are trained the _same way_ — just scaled up to billions of parameters and trained on massive datasets (books, web text, audio).

| Approach                  | Data Needed                    | Training Time              | Architecture         |
| ------------------------- | ------------------------------ | -------------------------- | -------------------- |
| From Scratch              | 20,000+ hours of labeled audio | Days/weeks on GPU clusters | Must design yourself |
| Pre-trained (HuggingFace) | Zero                           | Seconds (inference only)   | Already proven       |

**This section demonstrates inference** (generating music from a trained model). The _training_ used the exact same `.backward()` you just learned — just at scale.


In [ ]:
## ── Install HuggingFace dependencies ────────────────────────────────────────
## Run once; restart the kernel after installing if needed.

!pip install transformers accelerate scipy soundfile music21 --quiet


### Exercise — music generation

The next cell lets you choose between two pretrained models. For whichever you pick, change the prompt/seed and **predict** what the output will sound like before generating.


In [ ]:
## ── Model selector ──────────────────────────────────────────────────────────
## Change MUSIC_MODEL to switch between the two approaches at runtime.

# ── Pick one ─────────────────────────────────────────────────────────────────
MUSIC_MODEL = "musicgen"  # "musicgen"  |  "tunesformer"

# ── MusicGen config (used when MUSIC_MODEL == "musicgen") ────────────────────
#   Model sizes: musicgen-small (~300 MB) | musicgen-medium (~1.5 GB) | musicgen-large (~3.3 GB)
MUSICGEN_REPO = "facebook/musicgen-small"
MUSICGEN_PROMPT = "upbeat Irish folk music with fiddle and flute, lively jig"
MUSICGEN_DURATION = 8  # seconds of audio to generate

# ── TunesFormer config (used when MUSIC_MODEL == "tunesformer") ──────────────
#   TunesFormer generates ABC notation conditioned on a control-code seed.
#   Seed format:  X:<index>  T:<title>  M:<time sig>  K:<key>  then barlines.
TUNESFORMER_REPO = "sander-wood/tunesformer"
TUNESFORMER_SEED = "X:1\nT:My Generated Tune\nM:6/8\nL:1/8\nK:Gmaj\n|: G2A B2c |"
TUNESFORMER_NEW_TOKENS = 400  # max new tokens (≈ 1-2 full tunes)
TUNESFORMER_TEMPERATURE = 0.9
TUNESFORMER_TOP_K = 50

OUTPUT_WAV = "generated_music.wav"

print(f"Selected model: {MUSIC_MODEL!r}")

### Option A — `facebook/musicgen-small` (text-prompt → audio)

MusicGen is a Transformer encoder-decoder trained by Meta AI on 20 000 hours of
licensed music. You describe the music you want in plain English and it generates
a raw audio waveform directly — no ABC notation, no MIDI intermediate step.

Run the cell below when `MUSIC_MODEL = "musicgen"`.


In [ ]:
if MUSIC_MODEL == "musicgen":
    from transformers import pipeline
    import scipy.io.wavfile
    import numpy as np
    import IPython.display as ipd

    print(f"Loading {MUSICGEN_REPO} ...")
    musicgen_pipe = pipeline(
        "text-to-audio",
        model=MUSICGEN_REPO,
        device="cpu",  # change to 0 (or "cuda") if a GPU is available
    )

    print(
        f"Generating {MUSICGEN_DURATION}s of audio for prompt:\n  '{MUSICGEN_PROMPT}'"
    )
    result = musicgen_pipe(
        MUSICGEN_PROMPT,
        forward_params={
            "do_sample": True,
            "max_new_tokens": int(MUSICGEN_DURATION * 50),
        },
    )

    audio = result["audio"].squeeze()
    sr = result["sampling_rate"]

    # Normalise to int16 for WAV export
    audio_int16 = (audio / np.abs(audio).max() * 32767).astype(np.int16)
    scipy.io.wavfile.write(OUTPUT_WAV, sr, audio_int16)
    print(f"Saved to {OUTPUT_WAV}")

    ipd.display(ipd.Audio(audio, rate=sr))

### Option B — `sander-wood/tunesformer` (ABC seed → ABC notation)

TunesFormer is a GPT-style causal LM fine-tuned on thousands of Irish/Celtic folk
tunes in ABC notation. You supply a short **seed** (title, metre, key, a bar or
two) and the model completes the tune character-by-character — the same mechanism
as the MIT from-scratch LSTM, but using a pretrained model.

The output is **ABC notation text** which you can:

- Copy into [https://abc.rectanglered.com](https://abc.rectanglered.com) to hear it
- Convert to MIDI with `music21` (shown below)
- Convert to audio with `timidity` or GarageBand

Run the cell below when `MUSIC_MODEL = "tunesformer"`.


In [ ]:
if MUSIC_MODEL == "tunesformer":
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM

    print(f"Loading {TUNESFORMER_REPO} ...")
    tf_tokenizer = AutoTokenizer.from_pretrained(TUNESFORMER_REPO)
    lm_model = AutoModelForCausalLM.from_pretrained(TUNESFORMER_REPO)
    lm_model.eval()

    inputs = tf_tokenizer(TUNESFORMER_SEED, return_tensors="pt")

    print("Generating ABC notation ...")
    with torch.no_grad():
        output_ids = lm_model.generate(
            inputs["input_ids"],
            max_new_tokens=TUNESFORMER_NEW_TOKENS,
            do_sample=True,
            temperature=TUNESFORMER_TEMPERATURE,
            top_k=TUNESFORMER_TOP_K,
            pad_token_id=tf_tokenizer.eos_token_id,
        )

    generated_abc = tf_tokenizer.decode(output_ids[0], skip_special_tokens=True)
    print("\n-- Generated ABC notation ------------------------------------------")
    print(generated_abc)
    print("--------------------------------------------------------------------")

    # Optional: convert to MIDI with music21
    try:
        from music21 import converter, midi

        score = converter.parse(generated_abc, format="abc")
        mf = midi.translate.music21ObjectToMidiFile(score)
        midi_path = "generated_tune.mid"
        mf.open(midi_path, "wb")
        mf.write()
        mf.close()
        print(f"MIDI saved to {midi_path}")
    except Exception as e:
        print(f"[music21 MIDI export skipped: {e}]")
        print(
            "Tip: paste the ABC text above into https://abc.rectanglered.com to hear it."
        )

---

## Summary — What You Built

| Step | Concept                       | Key Idea                                                          | ✓   |
| ---- | ----------------------------- | ----------------------------------------------------------------- | --- |
| 1    | Tensors as Data Containers    | Scalars → vectors → matrices → batches; faster + safer than lists | ✓   |
| 2    | Operations on Tensors         | PyTorch builds computation graphs automatically                   | ✓   |
| 3    | The Manual Prediction Problem | Hand-tuning 3 weights fails; 100 features is impossible           | ✓   |
| 4    | Neural Networks in PyTorch    | nn.Module wraps learnable nn.Parameters                           | ✓   |
| 5    | Automatic Differentiation     | .backward() computes ∂loss/∂every_weight in one call              | ✓   |
| 6    | Gradient Descent in Action    | Iterative weight updates drive loss toward zero                   | ✓   |
| 7    | From Toy to Production        | Same autograd scales from 3 params to 1.5 billion (GPT-2)         | ✓   |
| 8    | Music Generation Bonus        | Pre-trained transformers (MusicGen, TunesFormer)                  | ✓   |

### Key Insights to Keep

- **Tensors are purpose-built**: GPU-ready, shape-safe, and 5-50× faster than lists for matrix operations.
- **nn.Module = learnable weights + forward pass**: The same pattern scales from 3 parameters to 1.5 billion.
- **Autograd is automatic calculus**: `.backward()` computes every ∂loss/∂param in one call, no manual derivatives needed.
- **Gradient descent is iterative refinement**: Each step nudges weights to reduce loss — cumulative tiny improvements converge to near-optimal.
- **Pre-trained models save months**: Training GPT-2 from scratch costs $50K+ in compute; inference on a pre-trained model takes seconds.
- **From toy to production, the machinery is identical**: If you understood gradient descent on house prices (3 weights), you understand it on Stable Diffusion (860M weights). Scale is the only difference.

**Next**: Dive into recurrent networks (Lab 2) and convolutional networks (Lab 3) — both built on the same PyTorch primitives you just mastered.
